# LangChain Tools and Agents Tutorial

This tutorial demonstrates how to use and combine **prebuilt tools** in LangChain (such as Wikipedia, ArXiv, and DuckDuckGo Search) and wrap them into a conversational **Agent** that can dynamically choose the best tool to answer your questions.

## Setup

In [1]:
# Load environment variables from .env file
from dotenv import load_dotenv
import os

load_dotenv()
print("Environment variables loaded successfully!")

Environment variables loaded successfully!


## 1. Wikipedia Prebuilt Tool

LangChain provides a built-in wrapper for querying Wikipedia. We configure it using the `WikipediaAPIWrapper` and package it as a query run tool.

In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Set up the API wrapper and tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

# Test running the tool directly
result = wiki_tool.run("Artificial Intelligence")
print(result)

C:\Users\eklav\AppData\Local\Temp\ipykernel_6328\835483252.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## 2. ArXiv Prebuilt Tool

The ArXiv database is home to scientific and academic papers. We can query it using `ArxivAPIWrapper`.

In [3]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

# Set up the Arxiv wrapper and tool
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=500)
arxiv_tool = ArxivQueryRun(arxiv_wrapper=arxiv_wrapper)

# Test running the tool directly
result = arxiv_tool.run("Attention Is All You Need")
print(result)

AttributeError: 'Search' object has no attribute 'results'

## 3. DuckDuckGo Search Tool

For searching the live internet, we use the `DuckDuckGoSearchRun` tool.

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

# Set up the search tool
search_tool = DuckDuckGoSearchRun()

# Test running the tool directly
result = search_tool.run("Google Gemini 2.5 Flash model release date")
print(result)

ImportError: Could not import ddgs python package. Please install it with `pip install -U ddgs`.

## 4. Creating a Tool-Calling Agent

We can combine these tools into a single list and initialize a LangChain agent using the Google Gemini model. The agent will read our query, determine which tool is best, invoke the tool, and give us a final answer.

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the LLM model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 2. Gather the tools list
tools = [wiki_tool, arxiv_tool, search_tool]

# 3. Create the chat prompt template for the agent
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful research assistant. Use your tools to answer user questions."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# 4. Initialize the agent
agent = create_tool_calling_agent(llm, tools, prompt)

# 5. Wrap the agent in the AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (C:\Users\eklav\Desktop\langchain\.venv\Lib\site-packages\langchain\agents\__init__.py)

### Test the Agent on Wikipedia

Let's ask a question that requires searching Wikipedia.

In [6]:
response = agent_executor.invoke({"input": "Who is the founder of Microsoft and what is his net worth?"})
print("\nFinal Answer:", response["output"])

NameError: name 'agent_executor' is not defined

### Test the Agent on ArXiv

Let's ask a question that requires searching scientific papers on ArXiv.

In [7]:
response = agent_executor.invoke({"input": "Summarize the paper 1706.03762 on ArXiv."})
print("\nFinal Answer:", response["output"])

NameError: name 'agent_executor' is not defined